## Hospital Patient Data Analysis

A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

In [1]:
### Firstly we load all the neccesarry libraries and datasets for analysis
import pandas as pd 
import numpy as np 

In [2]:
df = pd.read_csv(r"C:\Users\deytu\Desktop\study\EXCELR\Assignment\Data and Docs\Billing_Data.csv")
df.head()

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [3]:
df_1 = pd.read_csv(r"C:\Users\deytu\Desktop\study\EXCELR\Assignment\Data and Docs\Patient_Data.csv")
df_1

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


In [4]:
## Getting summary using .info() method 
df.info()
df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   PatientID         5 non-null      int64
 1   InsuranceCovered  5 non-null      int64
 2   FinalAmount       5 non-null      int64
dtypes: int64(3)
memory usage: 252.0 bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


#### We can observe that the billing data consisits of 5 patient ID's and has zero null values where as the patient data consists of 6 rows of patient data with 2 null values in BillAmount Column 

In [5]:
## Selecting only relevant columns for billing
df_bill = df_1[["PatientID","Department","Doctor" , "BillAmount" ]]
df_bill

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


In [6]:
## dropping administrative columns from the orignal dataset
df_1 = df_1.drop(columns = ["ReceptionistID","CheckInTime"])
df_1

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


In [7]:
## Grouping Data based on Department to get department wise bill ammount 
df_grouped = df_1.groupby("Department", as_index = False)["BillAmount"].sum()
df_grouped

,Department,BillAmount
0,Cardiology,16200.0
1,Dermatology,0.0
2,Neurology,0.0
3,Orthopedics,7500.0


In [8]:
## Removing Duplicate rows based on patient ids 
df_1 = df_1.drop_duplicates()
df_1

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


In [9]:
## Filling missing BillAmount with the mean bill amount 
df_1["BillAmount"] = df_1["BillAmount"].fillna(df_1["BillAmount"].mean())
df_1

C:\Users\deytu\AppData\Local\Temp\ipykernel_37240\928006867.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1["BillAmount"] = df_1["BillAmount"].fillna(df_1["BillAmount"].mean())


,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


In [10]:
## Merging the billing dataset with the patient dataset on Patient ID
df_merged = pd.merge(df_1 , df, on = "PatientID")
df_merged

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


In [11]:
### Using Row wise concatation using new row data 
### First creating new data 
new_patients = pd.DataFrame({
    "PatientID": [106, 107],
    "Name": ["Rahul", "Priya"],
    "Doctor": ["Dr. Smith", "Dr. John"],
    "Department": ["Cardiology", "Neurology"],
    "BillAmount": [8000, 15000]
})
new_patients

,PatientID,Name,Doctor,Department,BillAmount
0,106,Rahul,Dr. Smith,Cardiology,8000
1,107,Priya,Dr. John,Neurology,15000


In [12]:
## Now using concatation
df_conc1  = pd.concat([df_1 , new_patients ] , ignore_index = 1)
df_conc1

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333
5,106,Rahul,Cardiology,Dr. Smith,8000.000000
6,107,Priya,Neurology,Dr. John,15000.000000


In [13]:
##now using row wise concatation first we extract the two columns from patient data Insuracecoverd and finalAmount 
df_new_column = df[["InsuranceCovered","FinalAmount"]]
df_new_column 


,InsuranceCovered,FinalAmount
0,2000,3000
1,1500,3500
2,2500,5000
3,3000,3200
4,1000,4000


In [14]:
### noow concating the columns 
df_conc2 = pd.concat([df_1, df_new_column],  axis = 1)
df_conc2

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


### Outcomes 

In [19]:
# Final cleaned table with all missing and duplicate values handled which can futher be augmented and further analytics can be performed 
df_merged 

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000
